# Merge and audit translated game descriptions

Run this notebook only after the nine translated batch CSVs have been placed in `Translation Done`, alongside the `no_translation_needed.csv` created by `Description_Handling.ipynb`. The translated batches must retain the same columns and AppIDs as the input batches.

This notebook combines all ten files, restores the source row order, checks for duplicate/missing IDs and description-quality issues, and writes `Games_Description_English_Only.csv` to `Data Prep` only when the audit finds no empty descriptions, markup/emoji/mojibake artifacts, or confidently detected non-English segments. It does not print descriptions.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import regex as unicode_regex
from langdetect import detect_langs, DetectorFactory
from IPython.display import display

DetectorFactory.seed = 0
APP_ID_COLUMN = 'AppID'
NAME_COLUMN = 'Name'
DESCRIPTION_COLUMN = 'About the game'
NUM_BATCHES = 9
MIN_SEGMENT_LETTERS = 25
MIN_LANGUAGE_CONFIDENCE = 0.80

prep_candidates = [Path('.'), Path('Data Prep'), Path('../Data Prep')]
prep_dir = next((candidate for candidate in prep_candidates if (candidate / 'Translation Done').is_dir() and (candidate / 'games_semi_processed.csv').is_file()), None)
if prep_dir is None:
    raise FileNotFoundError('Could not locate Data Prep/Translation Done. Start Jupyter from the project root or Data Prep folder.')
translation_done_dir = prep_dir / 'Translation Done'
output_path = prep_dir / 'Games_Description_English_Only.csv'
print('Translation input folder:', translation_done_dir.resolve())
print('Final output:', output_path.resolve())

In [ ]:
# Load all translated batches plus the rows that never needed translation.
batch_paths = [translation_done_dir / f'translation_batch_{i:02d}_of_{NUM_BATCHES:02d}.csv' for i in range(1, NUM_BATCHES + 1)]
no_translation_path = translation_done_dir / 'no_translation_needed.csv'
missing_paths = [path for path in [*batch_paths, no_translation_path] if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing required translation output file(s): ' + ', '.join(path.name for path in missing_paths))

batch_frames = [pd.read_csv(path, encoding='utf-8-sig', low_memory=False, dtype={APP_ID_COLUMN: 'string'}) for path in batch_paths]
no_translation = pd.read_csv(no_translation_path, encoding='utf-8-sig', low_memory=False, dtype={APP_ID_COLUMN: 'string'})
all_frames = [*batch_frames, no_translation]
expected_columns = list(all_frames[0].columns)
for path, frame in zip([*batch_paths, no_translation_path], all_frames):
    if list(frame.columns) != expected_columns:
        raise ValueError(f'Column mismatch in {path.name}; each file must retain the same columns.')
required_columns = {APP_ID_COLUMN, NAME_COLUMN, DESCRIPTION_COLUMN, '__source_row'}
missing_columns = sorted(required_columns - set(expected_columns))
if missing_columns:
    raise KeyError(f'Required columns missing from translation files: {missing_columns}')

merged = pd.concat(all_frames, ignore_index=True)
if merged[APP_ID_COLUMN].isna().any():
    raise ValueError('Merged files contain missing AppIDs.')
duplicate_appids = merged.loc[merged[APP_ID_COLUMN].duplicated(keep=False), [APP_ID_COLUMN, NAME_COLUMN]]
if not duplicate_appids.empty:
    display(duplicate_appids.head(20))
    raise ValueError(f'Merged files contain {merged[APP_ID_COLUMN].duplicated().sum():,} duplicate AppIDs.')
if merged['__source_row'].isna().any() or merged['__source_row'].duplicated().any():
    raise ValueError('Source row markers are missing or duplicated; the input partition may be incomplete.')
merged['__source_row'] = pd.to_numeric(merged['__source_row'], errors='raise')
merged = merged.sort_values('__source_row', kind='stable').reset_index(drop=True)
print(f'Rows merged: {len(merged):,}')
print(f'Translated batch rows: {sum(len(frame) for frame in batch_frames):,}')
print(f'No-translation rows: {len(no_translation):,}')

In [ ]:
# Audit the combined descriptions before writing the definitive CSV.
HTML_TAG_RE = re.compile(r'</?\s*[A-Za-z][^>]*>')
HTML_ENTITY_RE = re.compile(r'&(?:[A-Za-z][A-Za-z0-9]+|#\d+|#x[0-9A-Fa-f]+);')
EMOJI_RE = unicode_regex.compile(r'(?:\p{Extended_Pictographic}(?:\p{Emoji_Modifier})?(?:\uFE0F|\uFE0E)?(?:\u200D\p{Extended_Pictographic}(?:\p{Emoji_Modifier})?(?:\uFE0F|\uFE0E)?)*)|(?:\p{Regional_Indicator}{2})|(?:[#*0-9]\uFE0F?\u20E3)')
MOJIBAKE_MARKERS = ('Ã', 'Â', 'â€', 'ðŸ', 'ï¿½')
MOJIBAKE_RE = re.compile('|'.join(re.escape(marker) for marker in MOJIBAKE_MARKERS))
CJK_RE = unicode_regex.compile(r'(?:\p{Script=Han}|\p{Script=Hiragana}|\p{Script=Katakana}|\p{Script=Hangul})')
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|(?<=[。！？])|[\r\n]+')
NON_ENGLISH_MARKERS = set('le les des est avec pour dans une jeu du et la los las del para pero juego una con que nicht der die das und ist ein spiel mit il lo gli che gioco per nao voce het een van zijn voor niet'.split())

def likely_needs_language_detection(text):
    text = '' if pd.isna(text) else str(text)
    if CJK_RE.search(text):
        return True
    letters = [char for char in text if char.isalpha()]
    non_ascii_letters = sum(ord(char) > 127 for char in letters)
    if non_ascii_letters >= 4 and non_ascii_letters / max(1, len(letters)) >= 0.02:
        return True
    words = set(unicode_regex.findall(r"\p{L}+", text.lower()))
    return len(words & NON_ENGLISH_MARKERS) >= 2

def analyze_language_segments(text):
    text = '' if pd.isna(text) else str(text)
    segments = [part.strip() for part in SENTENCE_SPLIT_RE.split(text) if part.strip()]
    total_letters = sum(char.isalpha() for part in segments for char in part)
    non_english_letters = 0
    non_english_segments = []
    for segment in segments:
        letters = sum(char.isalpha() for char in segment)
        if letters < MIN_SEGMENT_LETTERS:
            continue
        try:
            predictions = detect_langs(segment[:1500])
        except Exception:
            continue
        if predictions and predictions[0].prob >= MIN_LANGUAGE_CONFIDENCE and predictions[0].lang != 'en':
            non_english_letters += letters
            non_english_segments.append(segment)
    share = non_english_letters / total_letters if total_letters else 0.0
    return share, non_english_segments

texts = merged[DESCRIPTION_COLUMN].fillna('').astype(str)
residual_non_english = []
for position, text in enumerate(texts):
    if likely_needs_language_detection(text):
        share, segments = analyze_language_segments(text)
        if share > 0:
            residual_non_english.append((merged.iloc[position][APP_ID_COLUMN], merged.iloc[position][NAME_COLUMN], share, len(segments)))
    if (position + 1) % 5000 == 0:
        print(f'Audit progress: {position + 1:,}/{len(merged):,}')

audit_counts = {
    'rows': len(merged),
    'empty_descriptions': int(texts.str.strip().eq('').sum()),
    'emoji_matches': int(texts.map(lambda text: len(EMOJI_RE.findall(text))).sum()),
    'rows_with_html_tags': int(texts.map(lambda text: bool(HTML_TAG_RE.search(text))).sum()),
    'rows_with_html_entities': int(texts.map(lambda text: bool(HTML_ENTITY_RE.search(text))).sum()),
    'rows_with_mojibake_markers': int(texts.map(lambda text: bool(MOJIBAKE_RE.search(text))).sum()),
    'rows_with_replacement_characters': int(texts.str.contains('\ufffd', regex=False).sum()),
    'rows_with_non_ascii_letters': int(texts.map(lambda text: any(ord(char) > 127 and char.isalpha() for char in text)).sum()),
    'rows_with_confident_non_english_segments': len(residual_non_english),
}
print('Description audit:', json.dumps(audit_counts, ensure_ascii=False))
if residual_non_english:
    print('Review these AppID/name records; descriptions are omitted:')
    display(pd.DataFrame(residual_non_english[:50], columns=['AppID', 'Name', 'Non-English letter share', 'Non-English segment count']))

blocking_issues = {key: value for key, value in audit_counts.items() if key in {'empty_descriptions', 'emoji_matches', 'rows_with_html_tags', 'rows_with_html_entities', 'rows_with_mojibake_markers', 'rows_with_replacement_characters', 'rows_with_confident_non_english_segments'} and value}
if blocking_issues:
    raise RuntimeError('Audit found description issues; fix them before creating the English-only output: ' + json.dumps(blocking_issues, ensure_ascii=False))
print('Description audit passed.')

In [ ]:
# Save only after the merged dataset passes the description audit.
final = merged.drop(columns=['__source_row']).copy()
temp_output = output_path.with_name(output_path.stem + '.tmp.csv')
try:
    final.to_csv(temp_output, index=False, encoding='utf-8-sig')
    temp_output.replace(output_path)
except Exception:
    if temp_output.exists():
        temp_output.unlink()
    raise
print(f'Saved audited English-only dataset: {len(final):,} games')
print('Output:', output_path.resolve())